# EBM Two-Stage Recovery Model

Glass-box mirror of `xgboost_classification_regression_tuning.ipynb`, swapping XGBoost for InterpretML's Explainable Boosting Machine (EBM).

- **Stage 1**: `ExplainableBoostingClassifier` predicting `P(prob_recovered > 0)`
- **Stage 2**: `ExplainableBoostingRegressor` predicting `E(prob_recovered | prob_recovered > 0)` in **logit space** with sigmoid back-transform on inference
- **Combined**: `final_pred = p_nonzero * e_rate`

No hyperparameter tuning — defaults only. Goal is a baseline + glass-box exploration of feature contributions.

Designed to run on AWS SageMaker with S3 access.

In [ ]:
!pip install interpret polars

In [ ]:
import math
import time
import joblib
import boto3
import tempfile
import logging

import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score,
    mean_absolute_error, mean_squared_error, r2_score,
)

from interpret.glassbox import ExplainableBoostingClassifier, ExplainableBoostingRegressor
from interpret import show

In [ ]:
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(15)

## S3 helpers

In [ ]:
def save_model_to_s3(model, bucket_name, object_key):
    s3_client = boto3.client('s3')
    try:
        with tempfile.TemporaryFile() as fp:
            joblib.dump(model, fp)
            fp.seek(0)
            s3_client.put_object(Body=fp.read(), Bucket=bucket_name, Key=object_key)
            logging.info(f'{object_key} saved to s3 bucket {bucket_name}')
    except Exception as e:
        logging.exception(e)
        raise


def load_model_from_s3(bucket_name, object_key):
    s3_client = boto3.client('s3')
    try:
        with tempfile.TemporaryFile() as fp:
            s3_client.download_fileobj(bucket_name, object_key, fp)
            fp.seek(0)
            model = joblib.load(fp)
            logging.info(f'{object_key} loaded from s3 bucket {bucket_name}')
            return model
    except Exception as e:
        logging.exception(e)
        raise

## Data load + train/test split

In [ ]:
df = pl.read_parquet("s3://msds-26.2-data/clean/combined_recovery_data_aggregated_with_full_features.parquet")

df_train = df.filter(pl.col("year") < 2025)
df_test  = df.filter(pl.col("year") == 2025)

print(f"train rows: {len(df_train):,}  test rows: {len(df_test):,}")

## Feature definitions

Copied verbatim from `xgboost_classification_regression_tuning.ipynb`. If features change in one notebook, sync the others.

In [ ]:
identity_cols = [
    "hashed_fc", "gl_product_group", "week_date",
]

gl_composition_cols = [
    "share_food", "share_non_food", "share_pet_food",
    "share_RETAIL", "share_FBA", "share_hazmat",
]

gl_volume_cols = [
    "units_total", "cogs_total", "weight_total",
    "avg_cogs_per_unit", "avg_weight_per_unit", "cogs_per_unit_weight",
]

gl_at_site_cols = [
    "site_units_share_week", "site_weight_share_week",
]

site_context_cols = [
    "site_units_total_week", "site_weight_total_week",
    "site_type", "site_category", "country", "country_state",
]

temporal_site_context_cols = [
    'site_units_total_week_lag_1w', 'site_units_total_week_lag_4w',
    'site_units_total_week_lag_12w', 'site_units_total_week_lag_13w',
    'site_units_total_week_lag_52w',
    'site_weight_total_week_lag_1w', 'site_weight_total_week_lag_4w',
    'site_weight_total_week_lag_12w', 'site_weight_total_week_lag_13w',
    'site_weight_total_week_lag_52w',
    'site_prob_recovered_week_lag_1w', 'site_prob_recovered_week_lag_4w',
    'site_prob_recovered_week_lag_12w', 'site_prob_recovered_week_lag_13w',
    'site_prob_recovered_week_lag_52w',
    'site_prob_recovered_week_rolling_4w', 'site_prob_recovered_week_rolling_12w',
    'site_prob_recovered_week_rolling_26w', 'site_prob_recovered_week_rolling_52w',
]

calendar_cols = ["month", "week"]

temporal_composition_cols = [
    'share_RETAIL_lag_1w', 'share_RETAIL_lag_4w', 'share_RETAIL_lag_12w',
    'share_RETAIL_lag_13w', 'share_RETAIL_lag_52w',
    'share_FBA_lag_4w', 'share_FBA_lag_12w', 'share_FBA_lag_13w', 'share_FBA_lag_52w',
    'share_hazmat_lag_1w', 'share_hazmat_lag_4w', 'share_hazmat_lag_12w',
    'share_hazmat_lag_13w', 'share_hazmat_lag_52w',
    'share_food_lag_1w', 'share_food_lag_4w', 'share_food_lag_12w',
    'share_food_lag_13w', 'share_food_lag_52w',
    'share_non_food_lag_1w', 'share_non_food_lag_4w', 'share_non_food_lag_12w',
    'share_non_food_lag_13w', 'share_non_food_lag_52w',
    'share_pet_food_lag_1w', 'share_pet_food_lag_4w', 'share_pet_food_lag_12w',
    'share_pet_food_lag_13w', 'share_pet_food_lag_52w',
    'share_food_rolling_4w', 'share_food_rolling_12w',
    'share_non_food_rolling_4w', 'share_non_food_rolling_12w',
    'share_pet_food_rolling_4w', 'share_pet_food_rolling_12w',
    'share_RETAIL_rolling_4w', 'share_RETAIL_rolling_12w',
    'share_FBA_rolling_4w', 'share_FBA_rolling_12w',
    'share_hazmat_rolling_4w', 'share_hazmat_rolling_12w',
    'share_food_rolling_26w', 'share_food_rolling_52w',
    'share_non_food_rolling_26w', 'share_non_food_rolling_52w',
    'share_pet_food_rolling_26w', 'share_pet_food_rolling_52w',
    'share_RETAIL_rolling_26w', 'share_RETAIL_rolling_52w',
    'share_FBA_rolling_26w', 'share_FBA_rolling_52w',
    'share_hazmat_rolling_26w', 'share_hazmat_rolling_52w',
    'share_RETAIL_ewma_5a', 'share_RETAIL_ewma_1a',
    'share_FBA_ewma_5a', 'share_FBA_ewma_1a',
    'share_hazmat_ewma_5a', 'share_hazmat_ewma_1a',
    'share_food_ewma_5a', 'share_food_ewma_1a',
    'share_non_food_ewma_5a', 'share_non_food_ewma_1a',
    'share_pet_food_ewma_5a', 'share_pet_food_ewma_1a',
]

temporal_volume_cols = [
    'units_total_lag_1w', 'units_total_lag_4w', 'units_total_lag_12w',
    'units_total_lag_13w', 'units_total_lag_52w',
    'cogs_total_lag_1w', 'cogs_total_lag_4w', 'cogs_total_lag_12w',
    'cogs_total_lag_13w', 'cogs_total_lag_52w',
    'weight_total_lag_1w', 'weight_total_lag_4w', 'weight_total_lag_12w',
    'weight_total_lag_13w', 'weight_total_lag_52w',
    'units_total_rolling_4w', 'units_total_rolling_12w',
    'cogs_total_rolling_4w', 'cogs_total_rolling_12w',
    'weight_total_rolling_4w', 'weight_total_rolling_12w',
    'units_total_rolling_26w', 'units_total_rolling_52w',
    'cogs_total_rolling_26w', 'cogs_total_rolling_52w',
    'weight_total_rolling_26w', 'weight_total_rolling_52w',
    'units_total_ewma_5a', 'units_total_ewma_1a',
    'cogs_total_ewma_5a', 'cogs_total_ewma_1a',
    'weight_total_ewma_5a', 'weight_total_ewma_1a',
]

temporal_probability_cols = [
    'prob_recovered_lag_1w', 'prob_recovered_lag_4w', 'prob_recovered_lag_12w',
    'prob_recovered_lag_13w', 'prob_recovered_lag_52w',
    'prob_recovered_rolling_26w', 'prob_recovered_rolling_52w',
    'prob_recovered_rolling_4w', 'prob_recovered_rolling_12w',
    'prob_recovered_ewma_5a', 'prob_recovered_ewma_1a',
]

feature_cols = (
    gl_composition_cols + gl_volume_cols + gl_at_site_cols
    + site_context_cols + temporal_site_context_cols + calendar_cols
    + temporal_composition_cols + temporal_volume_cols + temporal_probability_cols
)

BASELINE_COLS = ["site_gl_mean_rate", "site_gl_std_rate", "site_gl_n_nonzero_weeks"]

print(f"Stage 1 features: {len(feature_cols)}")
print(f"Stage 2 features: {len(feature_cols) + len(BASELINE_COLS)}")

## Logit / categorical / sample-weight helpers

In [ ]:
EPS = 1e-6

def logit(p: np.ndarray) -> np.ndarray:
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))

def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-x))

In [ ]:
CAT_COLS = [
    'hashed_fc', 'gl_product_group', 'country',
    'country_state', 'site_type', 'site_category',
]

def cast_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    for col in CAT_COLS:
        if col in df.columns:
            df[col] = df[col].astype("category")
    return df

def build_feature_types(features: list[str]) -> list[str]:
    """EBM accepts feature_types=['nominal'|'continuous', ...] aligned to feature_names."""
    return ["nominal" if c in CAT_COLS else "continuous" for c in features]

In [ ]:
def compute_weights_stage2(y: np.ndarray) -> np.ndarray:
    """Sample weights emphasizing the 10-60% rate buckets."""
    w = np.ones(len(y))
    w[(y >= 0.1) & (y < 0.3)] = 3.0
    w[(y >= 0.3) & (y < 0.6)] = 8.0
    w[(y >= 0.6)]             = 2.0
    return w

In [ ]:
df_train = df_train.with_columns(
    pl.when(pl.col('prob_recovered') > 0).then(1).otherwise(0).alias('binary_recovered')
)
df_test = df_test.with_columns(
    pl.when(pl.col('prob_recovered') > 0).then(1).otherwise(0).alias('binary_recovered')
)

# Stage 1: EBM Classifier — `P(prob_recovered > 0)`

Default hyperparameters. EBM has no `scale_pos_weight` analogue; class imbalance is handled via boosting on residuals. We pass `feature_names` and `feature_types` explicitly so the glass-box explanations have the right labels.

In [ ]:
X_train_clf = cast_categoricals(df_train.select(feature_cols).to_pandas())
X_val_clf   = cast_categoricals(df_test.select(feature_cols).to_pandas())

y_train_clf = (df_train["prob_recovered"].to_pandas().values > 0).astype(np.int32)
y_val_clf   = (df_test["prob_recovered"].to_pandas().values > 0).astype(np.int32)

feature_types_stage1 = build_feature_types(feature_cols)

print(f"X_train: {X_train_clf.shape}  X_val: {X_val_clf.shape}")
print(f"positive rate (train): {y_train_clf.mean():.3f}  (val): {y_val_clf.mean():.3f}")

In [ ]:
start = time.perf_counter()

model_clf = ExplainableBoostingClassifier(
    feature_names=feature_cols,
    feature_types=feature_types_stage1,
    random_state=42,
)
model_clf.fit(X_train_clf, y_train_clf)

elapsed = (time.perf_counter() - start) / 60
print(f"Stage 1 fit time: {elapsed:.1f} min")

In [ ]:
p_nonzero = model_clf.predict_proba(X_val_clf)[:, 1]

print("Stage 1 metrics (default EBM):")
print(f"  AUC-ROC:    {roc_auc_score(y_val_clf, p_nonzero):.4f}")
print(f"  AUC-PR:     {average_precision_score(y_val_clf, p_nonzero):.4f}")
print(f"  F1 (t=0.5): {f1_score(y_val_clf, (p_nonzero >= 0.5).astype(int)):.4f}")

In [ ]:
thresholds = np.linspace(0.05, 0.95, 19)
rows = []
for t in thresholds:
    yhat = (p_nonzero >= t).astype(int)
    rows.append({
        "threshold": round(float(t), 2),
        "precision": precision_score(y_val_clf, yhat, zero_division=0),
        "recall":    recall_score(y_val_clf, yhat),
        "f1":        f1_score(y_val_clf, yhat),
    })
threshold_df = pl.DataFrame(rows)
best_threshold = float(threshold_df.sort("f1", descending=True)["threshold"][0])
print(f"Best F1 threshold: {best_threshold}")
print(threshold_df)

# Stage 2: EBM Regressor — `E(prob_recovered | > 0)` in logit space

Trains on non-zero rows only; sample weights emphasize the 10-60% rate band; target is logit-transformed and predictions are sigmoid-back-transformed before evaluation. Adds the `site_gl_baseline` features with a hierarchical prior fallback for unseen (hashed_fc, gl_product_group) pairs.

In [ ]:
def build_baseline_priors(df_train_nz: pl.DataFrame, target_col: str = "prob_recovered") -> dict:
    gl_baseline = (
        df_train_nz
        .group_by("gl_product_group")
        .agg([
            pl.col(target_col).mean().alias("gl_mean_rate"),
            pl.col(target_col).std().alias("gl_std_rate"),
        ])
    )
    site_baseline = (
        df_train_nz
        .group_by("hashed_fc")
        .agg([
            pl.col(target_col).mean().alias("site_mean_rate"),
            pl.col(target_col).std().alias("site_std_rate"),
        ])
    )
    return {
        "gl_baseline": gl_baseline,
        "site_baseline": site_baseline,
        "global_mean_rate": float(df_train_nz[target_col].mean()),
        "global_std_rate":  float(df_train_nz[target_col].std()),
    }


def fill_site_gl_baseline(df: pl.DataFrame, priors: dict) -> pl.DataFrame:
    df = df.join(priors["gl_baseline"],   on="gl_product_group", how="left")
    df = df.join(priors["site_baseline"], on="hashed_fc",        how="left")
    df = df.with_columns([
        pl.coalesce([
            pl.col("site_gl_mean_rate"),
            pl.col("gl_mean_rate"),
            pl.col("site_mean_rate"),
            pl.lit(priors["global_mean_rate"]),
        ]).alias("site_gl_mean_rate"),
        pl.coalesce([
            pl.col("site_gl_std_rate"),
            pl.col("gl_std_rate"),
            pl.col("site_std_rate"),
            pl.lit(priors["global_std_rate"]),
        ]).alias("site_gl_std_rate"),
        pl.col("site_gl_n_nonzero_weeks").fill_null(0).alias("site_gl_n_nonzero_weeks"),
    ])
    return df.drop(["gl_mean_rate", "gl_std_rate", "site_mean_rate", "site_std_rate"])


def holdout_site_gl_baseline(
    site_gl_baseline: pl.DataFrame,
    holdout_frac: float = 0.05,
    seed: int = 42,
) -> pl.DataFrame:
    all_pairs = site_gl_baseline.select(["hashed_fc", "gl_product_group"])
    holdout_pairs = (
        all_pairs
        .sample(fraction=holdout_frac, seed=seed)
        .with_columns(pl.lit(True).alias("_held_out"))
    )
    return (
        site_gl_baseline
        .join(holdout_pairs, on=["hashed_fc", "gl_product_group"], how="left")
        .filter(pl.col("_held_out").is_null())
        .drop("_held_out")
    )

In [ ]:
target_col = "prob_recovered"

df_train_nz = df_train.filter(pl.col(target_col) > 0)
df_test_nz  = df_test.filter(pl.col(target_col) > 0)

print(f"Stage 2 train: {len(df_train_nz):,}  test: {len(df_test_nz):,}")

site_gl_baseline = (
    df_train_nz
    .group_by(["hashed_fc", "gl_product_group"])
    .agg([
        pl.col(target_col).mean().alias("site_gl_mean_rate"),
        pl.col(target_col).std().alias("site_gl_std_rate"),
        pl.col(target_col).count().alias("site_gl_n_nonzero_weeks"),
    ])
)

priors = build_baseline_priors(df_train_nz, target_col=target_col)
HOLDOUT_FRAC = 0.05
site_gl_baseline_seen = holdout_site_gl_baseline(
    site_gl_baseline, holdout_frac=HOLDOUT_FRAC, seed=42,
)
print(f"Holdout pairs: {len(site_gl_baseline) - len(site_gl_baseline_seen):,} / "
      f"{len(site_gl_baseline):,} ({HOLDOUT_FRAC:.1%})")

df_train_nz = df_train_nz.join(
    site_gl_baseline_seen, on=["hashed_fc", "gl_product_group"], how="left",
)
df_train_nz = fill_site_gl_baseline(df_train_nz, priors)

df_test_nz = df_test_nz.join(
    site_gl_baseline, on=["hashed_fc", "gl_product_group"], how="left",
)
n_unseen = df_test_nz.filter(pl.col("site_gl_mean_rate").is_null()).height
print(f"Unseen site-GL pairs in test (pre-fill): {n_unseen:,} ({n_unseen/len(df_test_nz):.1%})")
df_test_nz = fill_site_gl_baseline(df_test_nz, priors)

extended_features = feature_cols + BASELINE_COLS
feature_types_stage2 = build_feature_types(extended_features)

X_train_reg = cast_categoricals(df_train_nz.select(extended_features).to_pandas())
X_val_reg   = cast_categoricals(df_test_nz.select(extended_features).to_pandas())

y_train_reg = df_train_nz[target_col].to_pandas().values
y_val_reg   = df_test_nz[target_col].to_pandas().values

y_train_logit = logit(y_train_reg)
y_val_logit   = logit(y_val_reg)

w_train_reg = compute_weights_stage2(y_train_reg)

print(f"\nX_train: {X_train_reg.shape}  X_val: {X_val_reg.shape}")

In [ ]:
start = time.perf_counter()

model_reg = ExplainableBoostingRegressor(
    feature_names=extended_features,
    feature_types=feature_types_stage2,
    random_state=42,
)
model_reg.fit(X_train_reg, y_train_logit, sample_weight=w_train_reg)

elapsed = (time.perf_counter() - start) / 60
print(f"Stage 2 fit time: {elapsed:.1f} min")

In [ ]:
preds_logit_nz = model_reg.predict(X_val_reg)
preds_nz       = np.clip(sigmoid(preds_logit_nz), 0.0, 1.0)

print("Stage 2 metrics (default EBM, non-zero rows only):")
print(f"  MAE:  {mean_absolute_error(y_val_reg, preds_nz):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_val_reg, preds_nz)):.4f}")
print(f"  R2:   {r2_score(y_val_reg, preds_nz):.4f}")

# Combined two-stage evaluation

`final_pred = p_nonzero * e_rate` over the full 2025 test set (zero and non-zero rows). Stratified MAE by `rate_bucket` and `volume_bucket` gives the standard slices used in the reference notebook.

In [ ]:
df_test_ext = df_test.join(
    site_gl_baseline, on=["hashed_fc", "gl_product_group"], how="left",
)
df_test_ext = fill_site_gl_baseline(df_test_ext, priors)

X_val_full_clf = cast_categoricals(df_test_ext.select(feature_cols).to_pandas())
X_val_full_reg = cast_categoricals(df_test_ext.select(extended_features).to_pandas())

p_nonzero_full = model_clf.predict_proba(X_val_full_clf)[:, 1]
e_rate_full    = np.clip(sigmoid(model_reg.predict(X_val_full_reg)), 0.0, 1.0)
preds_combined = p_nonzero_full * e_rate_full

y_val_full = df_test_ext[target_col].to_pandas().values

print("Combined two-stage metrics (full 2025 test):")
print(f"  MAE:  {mean_absolute_error(y_val_full, preds_combined):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_val_full, preds_combined)):.4f}")
print(f"  R2:   {r2_score(y_val_full, preds_combined):.4f}")

In [ ]:
residuals = preds_combined - y_val_full
diagnostics = df_test_ext.with_columns([
    pl.Series("y_true",       y_val_full),
    pl.Series("y_pred",       preds_combined),
    pl.Series("residual",     residuals),
    pl.Series("abs_residual", np.abs(residuals)),
]).with_columns([
    pl.when(pl.col("units_total") < 10).then(pl.lit("< 10"))
      .when(pl.col("units_total") < 100).then(pl.lit("10-100"))
      .when(pl.col("units_total") < 1000).then(pl.lit("100-1k"))
      .otherwise(pl.lit("> 1k"))
      .alias("volume_bucket"),
    pl.when(pl.col("y_true") == 0).then(pl.lit("zero"))
      .when(pl.col("y_true") < 0.1).then(pl.lit("0-10%"))
      .when(pl.col("y_true") < 0.3).then(pl.lit("10-30%"))
      .when(pl.col("y_true") < 0.6).then(pl.lit("30-60%"))
      .otherwise(pl.lit("> 60%"))
      .alias("rate_bucket"),
])

def stratified_mae(df: pl.DataFrame, group_col: str) -> pl.DataFrame:
    return (
        df.group_by(group_col)
        .agg([
            pl.col("abs_residual").mean().round(4).alias("mae"),
            pl.col("abs_residual").median().round(4).alias("median_mae"),
            pl.col("units_total").sum().alias("total_units"),
            pl.len().alias("n_rows"),
        ])
        .sort("mae", descending=True)
    )

print("── Combined MAE by rate bucket ──")
print(stratified_mae(diagnostics, "rate_bucket"))

print("\n── Combined MAE by volume bucket ──")
print(stratified_mae(diagnostics, "volume_bucket"))

print("\n── Combined MAE by GL (top 15) ──")
print(stratified_mae(diagnostics, "gl_product_group").head(15))

print("\n── Combined MAE by site type ──")
print(stratified_mae(diagnostics, "site_type"))

# Glass-box exploration

EBM is additive: `f(x) = intercept + Σ shape_i(x_i) + Σ pair_ij(x_i, x_j)`. Each shape function and detected pairwise interaction is directly inspectable, which is the whole point of using EBM here over XGBoost.

In [ ]:
def importance_table(model, top_n: int = 20) -> pl.DataFrame:
    explanation = model.explain_global()
    data = explanation.data()
    return (
        pl.DataFrame({"feature": data["names"], "importance": data["scores"]})
        .sort("importance", descending=True)
        .head(top_n)
    )

print("── Stage 1 (classifier) — top 20 features ──")
print(importance_table(model_clf, 20))

print("\n── Stage 2 (regressor, logit space) — top 20 features ──")
print(importance_table(model_reg, 20))

In [ ]:
# Interactive dashboards (render in-notebook on SageMaker).
show(model_clf.explain_global(name="Stage 1 — global"))
show(model_reg.explain_global(name="Stage 2 — global (logit space)"))

In [ ]:
def plot_shape(model, feature_name: str, ax=None, title_suffix: str = ""):
    """Plot the EBM shape function for a single feature in the model's native output space
    (logit for the regressor, log-odds for the classifier)."""
    explanation = model.explain_global()
    names = explanation.data()["names"]
    if feature_name not in names:
        print(f"  skip: {feature_name} not in model")
        return
    idx = names.index(feature_name)
    feat = explanation.data(idx)
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 3))
    if feat["type"] == "univariate":
        x = feat["names"]
        y = feat["scores"]
        if isinstance(x[0], (int, float)) and not isinstance(x[0], bool):
            ax.plot(x, y, marker="o", linewidth=1)
        else:
            ax.bar(range(len(x)), y)
            ax.set_xticks(range(len(x)))
            ax.set_xticklabels(x, rotation=45, ha="right")
    ax.axhline(0, color="grey", linestyle=":", linewidth=0.8)
    ax.set_title(f"{feature_name} {title_suffix}".strip())
    ax.set_xlabel(feature_name)
    ax.set_ylabel("contribution")


focus_features = [
    "site_gl_mean_rate",
    "site_gl_n_nonzero_weeks",
    "prob_recovered_lag_1w",
    "prob_recovered_ewma_5a",
    "share_food",
    "share_hazmat",
    "month",
    "week",
]

fig, axes = plt.subplots(4, 2, figsize=(14, 14))
for ax, feat in zip(axes.flat, focus_features):
    plot_shape(model_reg, feat, ax=ax, title_suffix="(Stage 2, logit)")
fig.suptitle("Stage 2 EBM shape functions", y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# EBM auto-detects pairwise interaction terms. List the strongest ones for each stage.
def interaction_table(model, top_n: int = 10) -> pl.DataFrame:
    data = model.explain_global().data()
    pairs = [(n, s) for n, s in zip(data["names"], data["scores"]) if "&" in n or " x " in n]
    if not pairs:
        return pl.DataFrame({"feature": [], "importance": []})
    return (
        pl.DataFrame({"feature": [p[0] for p in pairs], "importance": [p[1] for p in pairs]})
        .sort("importance", descending=True)
        .head(top_n)
    )

print("── Stage 1 — top pairwise interactions ──")
print(interaction_table(model_clf, 10))

print("\n── Stage 2 — top pairwise interactions ──")
print(interaction_table(model_reg, 10))

# Render the top-3 Stage-2 interaction heatmaps via the interpret dashboard.
show(model_reg.explain_global(name="Stage 2 — interactions"))

In [ ]:
# Pick 5 hand-selected rows from the 10-60% band — the slice that drives most of
# the Stage 2 weighted loss — for per-row local explanations.
y_val_arr = y_val_reg
mid_band  = np.where((y_val_arr >= 0.1) & (y_val_arr < 0.6))[0]
rng       = np.random.default_rng(42)
picks     = rng.choice(mid_band, size=min(5, len(mid_band)), replace=False)

X_picks = X_val_reg.iloc[picks].reset_index(drop=True)
y_picks = y_val_arr[picks]

local_exp = model_reg.explain_local(X_picks, y=logit(y_picks), name="Stage 2 — local")
show(local_exp)

# Print per-row predicted-vs-actual for context.
preds_picks = np.clip(sigmoid(model_reg.predict(X_picks)), 0.0, 1.0)
for i, (yi, pi) in enumerate(zip(y_picks, preds_picks)):
    print(f"  row {i}: y_true={yi:.3f}  y_pred={pi:.3f}")

# Persist to S3

Attach `site_gl_baseline_`, `baseline_priors_`, and the Stage-2 feature list to the regressor so downstream notebooks can load and predict without rebuilding the baseline.

In [ ]:
model_reg.site_gl_baseline_ = site_gl_baseline
model_reg.baseline_priors_  = priors
model_reg.feature_cols_stage2_ = extended_features

save_model_to_s3(model_clf, 'msds-26.2-data', 'model/ebm_classification_model.joblib')
save_model_to_s3(model_reg, 'msds-26.2-data', 'model/ebm_regression_model.joblib')

print("saved both EBM models to s3://msds-26.2-data/model/")